# Lab · Practice Project: mạng nơ-ron phân loại ung thư vú

Bài tập cho [`lesson.md`](lesson.md) (Module 7, Practice Project). Theo đúng lab "Practice Project: Neural Network for Breast Cancer Classification" của khóa: bộ dữ liệu **Breast Cancer Wisconsin (Diagnostic)**, mạng nơ-ron một lớp ẩn, `CrossEntropyLoss` + `Adam`, rồi ba bài luyện tập (đổi optimizer, đổi số nơ-ron, thử bộ dữ liệu Iris).

Bài này **không chấm điểm** trên Coursera, nhưng là bước chuẩn bị tốt nhất cho Final Project.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`. Bài 1 tải dữ liệu từ UCI nên cần có mạng.

## Chuẩn bị

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Bài 1. Tải dữ liệu Breast Cancer Wisconsin (Diagnostic)

Dùng `fetch_ucirepo(id=17)` (thư viện `ucimlrepo`). Lưu `X = ....data.features`, `y = ....data.targets` (cả hai là DataFrame).
Lưu `shapes = (X.shape, y.shape)` và `counts = y['Diagnosis'].value_counts().to_dict()`.
Câu hỏi: dữ liệu có cân bằng không? (`B` = lành tính, `M` = ác tính.)

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Bộ dữ liệu này nổi tiếng nên có sẵn trong kho UCI. Thư viện `ucimlrepo` lấy nó bằng một mã số.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`from ucimlrepo import fetch_ucirepo`; kết quả có thuộc tính `.data.features` và `.data.targets`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`ds = fetch_ucirepo(id=____)`; `X = ds.data.features`.

</details>

In [ ]:
from ucimlrepo import fetch_ucirepo

ds = None      # TODO
X = None       # TODO
y = None       # TODO
shapes = None  # TODO
counts = None  # TODO
print(shapes, counts)

In [ ]:
assert shapes == ((569, 30), (569, 1))
assert counts == {'B': 357, 'M': 212}, "357 lành tính, 212 ác tính: mất cân bằng"
print("✅ Bài 1 đúng")

## Bài 2. Cân bằng dữ liệu: 200 mẫu mỗi lớp

Gộp `X` và `y` thành một DataFrame `data` (`pd.concat([X, y], axis=1)`). Lấy ngẫu nhiên **200** mẫu lớp `B` và **200** mẫu lớp `M` (`DataFrame.sample(n=200, random_state=42)`), ghép lại thành `balanced_data`.
Lưu `balanced_counts = balanced_data['Diagnosis'].value_counts().to_dict()`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Tách theo lớp trước, lấy mẫu từng lớp, rồi nối lại.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`data[data['Diagnosis'] == 'B']`. Nối: `pd.concat([...])`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`data_M = data[data['Diagnosis'] == '____'].sample(n=200, random_state=42)`.

</details>

In [ ]:
data = None           # TODO
balanced_data = None  # TODO
balanced_counts = None  # TODO
print(balanced_counts)

In [ ]:
assert balanced_counts == {'B': 200, 'M': 200}
assert balanced_data.shape == (400, 31)
print("✅ Bài 2 đúng")

## Bài 3. Nhãn nhị phân và chia train/test

- `Xf = balanced_data.drop('Diagnosis', axis=1)`; `yf = balanced_data['Diagnosis'].map({'B': 0, 'M': 1})`.
- Chia 80/20 bằng `train_test_split(Xf, yf, test_size=0.2, random_state=42, stratify=yf)` vào `X_train, X_test, y_train, y_test`.
Câu hỏi: `stratify=yf` để làm gì?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Mạng cần nhãn là số. Và hai tập con phải có cùng tỉ lệ lớp với tập gốc.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`Series.map` với dictionary đổi chữ thành số. `stratify` giữ nguyên tỉ lệ lớp ở cả hai tập.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`X_train, X_test, y_train, y_test = train_test_split(Xf, yf, test_size=____, random_state=42, stratify=yf)`.

</details>

In [ ]:
Xf = None  # TODO
yf = None  # TODO
X_train, X_test, y_train, y_test = None, None, None, None  # TODO
print(X_train.shape, X_test.shape, y_test.value_counts().to_dict())

In [ ]:
assert X_train.shape == (320, 30) and X_test.shape == (80, 30)
assert y_test.value_counts().to_dict() == {0: 40, 1: 40}, "stratify giữ 40 + 40 ở tập test"
print("✅ Bài 3 đúng")

## Bài 4. Chuẩn hóa (chỉ học từ tập train) và tạo DataLoader

- Dùng `StandardScaler`: **`fit_transform` trên `X_train`**, và **chỉ `transform` trên `X_test`**.
- Đổi sang tensor: đặc trưng `float32`, nhãn `long` (số nguyên 64 bit; `CrossEntropyLoss` cần kiểu này).
- `TensorDataset` rồi `train_loader` (`batch_size=2`, `shuffle=True`) và `test_loader` (`batch_size=2`, `shuffle=False`).
Lưu `len_train_loader`, `len_test_loader`.
Câu hỏi: vì sao không được `fit` scaler trên tập test?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Tập test đại diện cho dữ liệu chưa từng thấy. Thông tin từ nó không được phép lọt vào bước chuẩn bị.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`scaler.fit_transform(X_train)` rồi `scaler.transform(X_test)`. `torch.tensor(..., dtype=torch.float32)`; nhãn: `torch.tensor(y_train.values, dtype=torch.long)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=2, shuffle=True)`.

</details>

In [ ]:
scaler = StandardScaler()
X_train_t = None  # TODO
X_test_t = None   # TODO
y_train_t = None  # TODO
y_test_t = None   # TODO
train_loader = None  # TODO
test_loader = None   # TODO
len_train_loader = None  # TODO
len_test_loader = None   # TODO
print(len_train_loader, len_test_loader)

In [ ]:
assert X_train_t.dtype == torch.float32 and y_train_t.dtype == torch.long
assert torch.allclose(X_train_t.mean(dim=0), torch.zeros(30), atol=1e-4), "train: trung bình 0 từng cột"
assert not torch.allclose(X_test_t.mean(dim=0), torch.zeros(30), atol=1e-4), "test: KHÔNG chính xác 0, vì dùng thống kê của train"
assert len_train_loader == 160 and len_test_loader == 40
print("✅ Bài 4 đúng")

## Bài 5. Mạng nơ-ron một lớp ẩn

Viết `ClassificationNet(nn.Module)` với `__init__(self, input_units=30, hidden_units=64, output_units=2)`: `self.fc1 = nn.Linear(input_units, hidden_units)`, `self.fc2 = nn.Linear(hidden_units, output_units)`; `forward`: `relu` sau `fc1`, rồi `fc2` (không sigmoid).
Tạo `model = ClassificationNet()`. Lưu:
- `out_shape`: shape đầu ra khi đưa vào một lô 2 mẫu (`X_train_t[:2]`);
- `n_params`: tổng số tham số.
Tính tay `n_params` trước.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Hai lớp tuyến tính, giữa chúng là một hàm kích hoạt. Lớp cuối cho **hai điểm số** (một cho mỗi lớp), chưa phải xác suất.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.relu`. Số tham số mỗi `Linear(a, b)` là `a·b + b`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`30·64 + 64 + 64·2 + 2`.

</details>

In [ ]:
class ClassificationNet(nn.Module):
    pass  # TODO

model = None      # TODO
out_shape = None  # TODO
n_params = None   # TODO
print(out_shape, n_params)

In [ ]:
assert out_shape == (2, 2)
assert n_params == 30 * 64 + 64 + 64 * 2 + 2 == 2114
print("✅ Bài 5 đúng: đây là mạng nơ-ron đầu tiên của bạn")

## Bài 6. Hàm loss, optimizer và loss ban đầu

- `criterion = nn.CrossEntropyLoss()` và `optimizer = optim.Adam(model.parameters(), lr=0.001)`.
- `loss_init`: loss của mô hình **chưa huấn luyện** trên toàn bộ tập test (dưới `torch.no_grad()`).
Đoán trước: với hai lớp và dự đoán gần như ngẫu nhiên, loss khoảng bao nhiêu? (Gợi ý: `ln 2`.)

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

`CrossEntropyLoss` nhận **điểm số thô** (logit) và nhãn là chỉ số lớp. Nó tự áp dụng softmax bên trong.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`criterion(model(X_test_t), y_test_t).item()`. Nếu mô hình dự đoán 50/50 thì loss là `−ln(0.5)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`with torch.no_grad(): loss_init = criterion(model(X_test_t), y_test_t).item()`.

</details>

In [ ]:
criterion = None  # TODO
optimizer = None  # TODO
loss_init = None  # TODO
print(loss_init)

In [ ]:
assert 0.4 < loss_init < 1.2, "mô hình ngẫu nhiên: loss gần ln 2 ≈ 0.693"
print("✅ Bài 6 đúng")

## Bài 7. Huấn luyện 10 epoch, theo dõi train và test

Viết `fit(model, optimizer, epochs=10)` làm cho mỗi epoch:
- `model.train()`, duyệt `train_loader` (zero_grad, forward, loss, backward, step), loss epoch = trung bình loss các lô;
- `model.eval()` và `torch.no_grad()`, duyệt `test_loader`, loss test = trung bình loss các lô.
Trả về `(train_losses, test_losses)` (mỗi cái là list độ dài `epochs`).
Chạy với `model`, `optimizer` ở bài 6. Rồi tính `test_acc`: độ chính xác của `model` trên `X_test_t` (nhãn dự đoán = chỉ số điểm số lớn nhất: `argmax(dim=1)`).

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Vòng lặp có hai pha. Hai lệnh `model.train()` và `model.eval()` báo cho mô hình biết nó đang ở pha nào.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Loss mỗi epoch: `running_loss / len(train_loader)`. Độ chính xác: `(model(X_test_t).argmax(dim=1) == y_test_t).float().mean()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Trong pha test, bọc vòng lặp bằng `with torch.no_grad():` và **không** gọi `backward()`.

</details>

In [ ]:
def fit(model, optimizer, epochs=10):
    pass  # TODO

train_losses, test_losses = None, None  # TODO
test_acc = None                          # TODO
print(train_losses, test_losses, test_acc)

In [ ]:
assert len(train_losses) == len(test_losses) == 10
assert train_losses[-1] < train_losses[0], "train loss phải giảm"
assert test_losses[-1] < loss_init
assert test_acc > 0.93, f"độ chính xác test nên > 93%, đang là {test_acc:.3f}"
print(f"✅ Bài 7 đúng. Độ chính xác test: {test_acc:.1%}")

## Bài 8. Vẽ đường loss

Vẽ `train_losses` (liền) và `test_losses` (nét đứt) theo epoch trên cùng một đồ thị, có nhãn trục và chú giải.
Lưu `gap = test_losses[-1] - train_losses[-1]` và nhận xét: có dấu hiệu overfitting không? (Gợi ý: hai đường có tách xa nhau không, test loss có tăng lên không?)

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Điều cần nhìn: hai đường giảm cùng nhau hay tách nhau, và đường test có quay đầu đi lên không.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`plt.plot(range(1, 11), train_losses, label=...)` và `linestyle='--'` cho đường test.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Đừng quên `plt.xlabel`, `plt.ylabel`, `plt.legend()` và `plt.show()`.

</details>

In [ ]:
gap = None  # TODO
# TODO: vẽ
print(gap)

In [ ]:
assert isinstance(gap, float)
assert abs(gap) < 0.3, "hai đường loss gần nhau: chưa overfit"
print("✅ Bài 8 đúng")

## Bài 9. Luyện tập 1: đổi sang optimizer SGD

Tạo mô hình **mới** `model_sgd = ClassificationNet()` và optimizer `optim.SGD(..., lr=0.001, momentum=0.9, weight_decay=0.0001)`.
Huấn luyện 10 epoch bằng `fit`. Lưu `acc_sgd` (độ chính xác test) và `loss_sgd = test_losses_sgd[-1]`.
Câu hỏi: `momentum` và `weight_decay` làm gì? So sánh với Adam ở bài 7.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Chỉ đổi dòng tạo optimizer; mọi thứ khác dùng lại.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`optim.SGD(model_sgd.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0001)`. Hàm `fit` đã nhận optimizer làm tham số.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`_, test_losses_sgd = fit(model_sgd, optimizer_sgd)`.

</details>

In [ ]:
model_sgd = None      # TODO
optimizer_sgd = None  # TODO
# TODO: huấn luyện
acc_sgd = None   # TODO
loss_sgd = None  # TODO
print(acc_sgd, loss_sgd)

In [ ]:
assert acc_sgd > 0.85
assert loss_sgd < loss_init
print(f"✅ Bài 9 đúng (SGD+momentum: {acc_sgd:.1%})")

## Bài 10. Luyện tập 2: đổi số nơ-ron ở lớp ẩn

Tạo `model_small = ClassificationNet(hidden_units=16)` với Adam `lr=0.001`, huấn luyện 10 epoch. Lưu `n_params_small` (tính tay trước) và `acc_small`.
Câu hỏi: ít nơ-ron hơn thì độ chính xác thay đổi thế nào, số tham số giảm bao nhiêu?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Số tham số phụ thuộc vào kích thước lớp ẩn ở cả hai lớp `Linear`.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`30·16 + 16 + 16·2 + 2`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`optim.Adam(model_small.parameters(), lr=0.001)`.

</details>

In [ ]:
model_small = None  # TODO
n_params_small = None  # TODO
acc_small = None       # TODO
print(n_params_small, acc_small)

In [ ]:
assert n_params_small == 30 * 16 + 16 + 16 * 2 + 2 == 530
assert acc_small > 0.9
print(f"✅ Bài 10 đúng (16 nơ-ron: {acc_small:.1%}, chỉ {n_params_small} tham số)")

## Bài 11. Luyện tập 3: bộ dữ liệu Iris (3 lớp)

Dùng `sklearn.datasets.load_iris()` (150 mẫu, 4 đặc trưng, **3 lớp**). Làm lại toàn bộ quy trình: chia 80/20 (`random_state=42`, `stratify`), chuẩn hóa (fit trên train), tensor, DataLoader (`batch_size=8`), mạng `ClassificationNet(input_units=4, hidden_units=16, output_units=3)`, `CrossEntropyLoss`, Adam `lr=0.01`, huấn luyện 30 epoch.
Lưu `acc_iris` (độ chính xác trên tập test 30 mẫu).
Điểm mới so với phần ung thư vú: số lớp đầu ra là 3.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Quy trình giống hệt. Chỉ ba con số đổi: số đặc trưng vào, số lớp ra, và kích thước dữ liệu.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`CrossEntropyLoss` làm việc với bất kỳ số lớp nào miễn nhãn là chỉ số `0..K-1` kiểu `long`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`load_iris(return_X_y=True)` cho `(X, y)` dạng NumPy.

</details>

In [ ]:
from sklearn.datasets import load_iris

acc_iris = None  # TODO
print(acc_iris)

In [ ]:
assert acc_iris > 0.9, f"Iris dễ phân loại: nên > 90%, đang là {acc_iris:.3f}"
print(f"✅ Bài 11 đúng (Iris: {acc_iris:.1%})")

## Xong rồi
Đối chiếu với `solution/`, rồi quay lại `lesson.md` nếu còn chỗ chưa chắc.